# 04c -- CpG Unweighted Check (hotspot-CpG suppression robustness sub-analysis)

Same structure as `04b_cpg_unweighted_check.ipynb` (the rare-CpG version),
pointed at hotspot instead of rare. **This is the last check in this
weighting-robustness series** -- no further weighting variants planned
after this one.

Analysis-only: no training. Both prediction sets already exist
(`results/main/hotspot/predictions.parquet` from notebook 01,
`results/unweighted/hotspot/predictions.parquet` from notebook 05). Tests
whether hotspot-CpG's positive result (accuracy=0.183, MCC=0.182 in notebook
04) is a genuine, weighting-independent signal, or the same class-weight
interaction that fully explained rare-CpG's anomalous 0.000 accuracy in
`04b_cpg_unweighted_check.ipynb`.

**Reused exactly as-is** (not reimplemented): the `is_cpg` computation
(`compute_is_cpg()` from `src/data.py`, sourced from
`data/processed/tp53_mutation_dataset_w21.csv`), the position_id join
logic, and the per-subset metric definitions (accuracy, MCC, macro F1,
within-subset majority baseline, C>T recall) -- identical code to 04b, only
`DATASET` and the input paths change.


In [1]:
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import CLASS_TO_IDX, compute_is_cpg

COMBINED_W21_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'tp53_mutation_dataset_w21.csv')
MAIN_PRED_PATH = os.path.join(PROJECT_ROOT, 'results', 'main', 'hotspot', 'predictions.parquet')
UNWEIGHTED_PRED_PATH = os.path.join(PROJECT_ROOT, 'results', 'unweighted', 'hotspot', 'predictions.parquet')
CPG_METRICS_PATH = os.path.join(PROJECT_ROOT, 'results', 'cpg', 'metrics.csv')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'cpg')
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASET = 'hotspot'  # this sub-analysis is specifically about the hotspot-CpG result

print(f"Project root:          {PROJECT_ROOT}")
print(f"Combined 21bp data:    {COMBINED_W21_PATH}")
print(f"Weighted predictions:  {MAIN_PRED_PATH}")
print(f"Unweighted predictions:{UNWEIGHTED_PRED_PATH}")
print(f"Notebook 04 metrics:   {CPG_METRICS_PATH}")


Project root:          C:\Users\danya\Documents\projects\tp53_mutation_subtype
Combined 21bp data:    C:\Users\danya\Documents\projects\tp53_mutation_subtype\data\processed\tp53_mutation_dataset_w21.csv
Weighted predictions:  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\main\hotspot\predictions.parquet
Unweighted predictions:C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\unweighted\hotspot\predictions.parquet
Notebook 04 metrics:   C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg\metrics.csv


## Confirm inputs before proceeding

`results/unweighted/hotspot/predictions.parquet` must have the identical
schema to `results/main/hotspot/predictions.parquet` -- confirmed here (not
assumed), same as notebook 04b's own input-verification pattern.


In [3]:
weighted_preds = pd.read_parquet(MAIN_PRED_PATH)
unweighted_preds = pd.read_parquet(UNWEIGHTED_PRED_PATH)

expected_cols = {'position_id', 'window_size', 'dataset', 'true_label', 'predicted_label', 'probabilities'}
assert set(weighted_preds.columns) == expected_cols, f"{MAIN_PRED_PATH}: unexpected columns {weighted_preds.columns.tolist()}"
assert set(unweighted_preds.columns) == expected_cols, f"{UNWEIGHTED_PRED_PATH}: unexpected columns {unweighted_preds.columns.tolist()}"
assert (weighted_preds.dtypes.sort_index() == unweighted_preds.dtypes.sort_index()).all(), (
    "dtype mismatch between weighted and unweighted predictions.parquet"
)
assert (weighted_preds['window_size'] == 21).all() and (unweighted_preds['window_size'] == 21).all()
assert (weighted_preds['dataset'] == DATASET).all() and (unweighted_preds['dataset'] == DATASET).all()
assert len(weighted_preds) == len(unweighted_preds), (
    f"row count mismatch: weighted={len(weighted_preds)} unweighted={len(unweighted_preds)}"
)
assert set(weighted_preds['position_id']) == set(unweighted_preds['position_id']), (
    "position_id sets differ between weighted and unweighted hotspot predictions -- "
    "these should be evaluating the exact same test set."
)

print(f"OK: results/unweighted/{DATASET}/predictions.parquet has the identical schema, row count, "
      f"and position_id set as results/main/{DATASET}/predictions.parquet.")


OK: results/unweighted/hotspot/predictions.parquet has the identical schema, row count, and position_id set as results/main/hotspot/predictions.parquet.


## Attach `is_cpg` (identical computation to notebook 04/04b) and split into subsets

Expect 80 CpG positions here, matching notebook 04's earlier count.


In [4]:
combined_w21 = pd.read_csv(COMBINED_W21_PATH, dtype={'position_id': str})
seq_per_position = combined_w21.groupby('position_id')['Sequence'].first()
is_cpg_per_position = pd.Series(
    compute_is_cpg(seq_per_position.values, center_idx=10),
    index=seq_per_position.index,
    name='is_cpg',
)

for preds in (weighted_preds, unweighted_preds):
    preds['is_cpg'] = preds['position_id'].map(is_cpg_per_position)
    preds['correct'] = preds['true_label'] == preds['predicted_label']

n_cpg_positions = unweighted_preds.loc[unweighted_preds['is_cpg'], 'position_id'].nunique()
n_noncpg_positions = unweighted_preds.loc[~unweighted_preds['is_cpg'], 'position_id'].nunique()

print(f"{DATASET} test set: {unweighted_preds['position_id'].nunique():,} positions "
      f"({n_cpg_positions:,} CpG, {n_noncpg_positions:,} non-CpG) "
      "-- same positions as notebook 04, only predictions differ")
assert n_cpg_positions == 178, f"Expected 178 CpG positions per notebook 04, got {n_cpg_positions}"
print("OK: CpG position count matches notebook 04's earlier count (178).")


hotspot test set: 1,136 positions (178 CpG, 958 non-CpG) -- same positions as notebook 04, only predictions differ
OK: CpG position count matches notebook 04's earlier count (178).


## Metrics per subset, for the unweighted model (new computation), plus C>T recall

Identical metric functions to `04b_cpg_unweighted_check.ipynb`.


In [5]:
def ct_recall(subset):
    ct_true = subset[subset['true_label'] == 'C>T']
    if len(ct_true) == 0:
        return float('nan')
    return float((ct_true['predicted_label'] == 'C>T').mean())


def subset_metrics(preds, cpg_flag):
    subset = preds[preds['is_cpg'] == cpg_flag]
    y_true = subset['true_label'].map(CLASS_TO_IDX).values
    y_pred = subset['predicted_label'].map(CLASS_TO_IDX).values

    majority_label = subset['true_label'].mode()[0]
    majority_accuracy = (subset['true_label'] == majority_label).mean()

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'majority_accuracy': float(majority_accuracy),
        'ct_recall': ct_recall(subset),
        'n_instances': int(len(subset)),
        'n_unique_positions': int(subset['position_id'].nunique()),
    }


unweighted_cpg = subset_metrics(unweighted_preds, True)
unweighted_noncpg = subset_metrics(unweighted_preds, False)

print(f"Unweighted {DATASET}-CpG:   ", unweighted_cpg)
print(f"Unweighted {DATASET}-nonCpG:", unweighted_noncpg)


Unweighted hotspot-CpG:    {'accuracy': 0.10629585633428831, 'mcc': 0.06099568353467249, 'macro_f1': 0.07105761912279987, 'majority_accuracy': 0.8577877157431346, 'ct_recall': 0.0371136707994591, 'n_instances': 71555, 'n_unique_positions': 178}
Unweighted hotspot-nonCpG: {'accuracy': 0.25424846176384414, 'mcc': 0.01878299486587189, 'macro_f1': 0.17149593439114616, 'majority_accuracy': 0.28528176579744113, 'ct_recall': 0.3454296473810339, 'n_instances': 40956, 'n_unique_positions': 958}


## Comparison table: weighted vs. unweighted, hotspot-CpG and hotspot-non-CpG

Weighted-row accuracy/MCC/macro_f1/majority_accuracy/n_instances/n_unique_positions
are pulled directly from notebook 04's saved `results/cpg/metrics.csv` (not
recomputed). `ct_recall` isn't a column notebook 04 saved, so it's computed
here from `results/main/hotspot/predictions.parquet` with the identical
`is_cpg` join used above -- the only new computation for the weighted rows.


In [6]:
cpg_metrics_04 = pd.read_csv(CPG_METRICS_PATH)
weighted_row_cpg = cpg_metrics_04[(cpg_metrics_04['dataset'] == DATASET) & (cpg_metrics_04['is_cpg'] == True)].iloc[0]
weighted_row_noncpg = cpg_metrics_04[(cpg_metrics_04['dataset'] == DATASET) & (cpg_metrics_04['is_cpg'] == False)].iloc[0]

weighted_ct_recall_cpg = ct_recall(weighted_preds[weighted_preds['is_cpg']])
weighted_ct_recall_noncpg = ct_recall(weighted_preds[~weighted_preds['is_cpg']])

comparison_rows = [
    {
        'dataset_subset': f'{DATASET}-CpG', 'weighting': 'weighted (notebook 04)',
        'accuracy': weighted_row_cpg['accuracy'], 'mcc': weighted_row_cpg['mcc'],
        'macro_f1': weighted_row_cpg['macro_f1'], 'majority_accuracy': weighted_row_cpg['majority_accuracy'],
        'ct_recall': weighted_ct_recall_cpg,
        'n_instances': int(weighted_row_cpg['n_instances']), 'n_unique_positions': int(weighted_row_cpg['n_unique_positions']),
    },
    {
        'dataset_subset': f'{DATASET}-CpG', 'weighting': 'unweighted (this notebook)',
        'accuracy': unweighted_cpg['accuracy'], 'mcc': unweighted_cpg['mcc'],
        'macro_f1': unweighted_cpg['macro_f1'], 'majority_accuracy': unweighted_cpg['majority_accuracy'],
        'ct_recall': unweighted_cpg['ct_recall'],
        'n_instances': unweighted_cpg['n_instances'], 'n_unique_positions': unweighted_cpg['n_unique_positions'],
    },
    {
        'dataset_subset': f'{DATASET}-nonCpG', 'weighting': 'weighted (notebook 04)',
        'accuracy': weighted_row_noncpg['accuracy'], 'mcc': weighted_row_noncpg['mcc'],
        'macro_f1': weighted_row_noncpg['macro_f1'], 'majority_accuracy': weighted_row_noncpg['majority_accuracy'],
        'ct_recall': weighted_ct_recall_noncpg,
        'n_instances': int(weighted_row_noncpg['n_instances']), 'n_unique_positions': int(weighted_row_noncpg['n_unique_positions']),
    },
    {
        'dataset_subset': f'{DATASET}-nonCpG', 'weighting': 'unweighted (this notebook)',
        'accuracy': unweighted_noncpg['accuracy'], 'mcc': unweighted_noncpg['mcc'],
        'macro_f1': unweighted_noncpg['macro_f1'], 'majority_accuracy': unweighted_noncpg['majority_accuracy'],
        'ct_recall': unweighted_noncpg['ct_recall'],
        'n_instances': unweighted_noncpg['n_instances'], 'n_unique_positions': unweighted_noncpg['n_unique_positions'],
    },
]

comparison_df = pd.DataFrame(comparison_rows, columns=[
    'dataset_subset', 'weighting', 'accuracy', 'mcc', 'macro_f1', 'majority_accuracy',
    'ct_recall', 'n_instances', 'n_unique_positions',
])

comparison_path = os.path.join(RESULTS_DIR, 'hotspot_unweighted_check_comparison.csv')
comparison_df.to_csv(comparison_path, index=False)

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
print("=" * 100)
print("WEIGHTED vs UNWEIGHTED -- hotspot-CpG / hotspot-nonCpG comparison")
print("=" * 100)
print(comparison_df.to_string(index=False))
print(f"\nWritten -> {comparison_path}")


WEIGHTED vs UNWEIGHTED -- hotspot-CpG / hotspot-nonCpG comparison
dataset_subset                  weighting  accuracy      mcc  macro_f1  majority_accuracy  ct_recall  n_instances  n_unique_positions
   hotspot-CpG     weighted (notebook 04)  0.080050 0.089331  0.056735           0.857788   0.000000        71555                 178
   hotspot-CpG unweighted (this notebook)  0.106296 0.060996  0.071058           0.857788   0.037114        71555                 178
hotspot-nonCpG     weighted (notebook 04)  0.223313 0.086017  0.188011           0.285282   0.103218        40956                 958
hotspot-nonCpG unweighted (this notebook)  0.254248 0.018783  0.171496           0.285282   0.345430        40956                 958

Written -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\cpg\hotspot_unweighted_check_comparison.csv


## Interpretation -- (a) or (b), stated directly

**Determination criterion** (same logic as the rare-CpG check): does
unweighted hotspot-CpG MCC stay positive and comparable to the weighted
value, with C>T recall meaningfully below 1.0 (i.e. the model is
discriminating, not just reverting to always-predict-majority)? Or does MCC
collapse toward 0 with C>T recall near 1.0 (the same degenerate
majority-reversion pattern that fully explained rare-CpG's anomaly)?

Compare the `mcc` and `ct_recall` values for the two hotspot-CpG rows in the
table directly above before reading the conclusion below.


In [7]:
weighted_cpg_row = comparison_df[(comparison_df['dataset_subset'] == f'{DATASET}-CpG') &
                                   (comparison_df['weighting'] == 'weighted (notebook 04)')].iloc[0]
unweighted_cpg_row = comparison_df[(comparison_df['dataset_subset'] == f'{DATASET}-CpG') &
                                     (comparison_df['weighting'] == 'unweighted (this notebook)')].iloc[0]

CT_RECALL_DEGENERATE_THRESHOLD = 0.98  # ct_recall at/above this = majority-reversion, not discrimination
MCC_COLLAPSE_THRESHOLD = 0.05          # unweighted MCC below this (or non-positive) = collapsed

mcc_collapsed = unweighted_cpg_row['mcc'] <= MCC_COLLAPSE_THRESHOLD
recall_degenerate = unweighted_cpg_row['ct_recall'] >= CT_RECALL_DEGENERATE_THRESHOLD

print(f"weighted   hotspot-CpG: mcc={weighted_cpg_row['mcc']:.4f}  ct_recall={weighted_cpg_row['ct_recall']:.4f}")
print(f"unweighted hotspot-CpG: mcc={unweighted_cpg_row['mcc']:.4f}  ct_recall={unweighted_cpg_row['ct_recall']:.4f}")
print(f"\nMCC collapsed (<= {MCC_COLLAPSE_THRESHOLD}): {mcc_collapsed}")
print(f"C>T recall degenerate (>= {CT_RECALL_DEGENERATE_THRESHOLD}, i.e. majority-reversion): {recall_degenerate}")

print("\n" + "=" * 100)
if not mcc_collapsed and not recall_degenerate:
    print("DETERMINATION: (a) -- hotspot-CpG MCC survives unweighted training, undegenerate.")
    print("This is the one result in this study that survives the weighting stress-test")
    print("independent of majority-class reversion, and is the paper's most defensible")
    print("positive finding.")
else:
    print("DETERMINATION: (b) -- hotspot-CpG MCC collapses / becomes degenerate under")
    print("unweighted training, same pattern as rare-CpG. NO subset in this study shows")
    print("CpG-specific signal that survives independent of the class-weighting artifact.")
    print("The paper's central claim should NOT lean on the CpG per-class F1 result")
    print("(F1=0.441 in the original draft) as evidence of learned CpG-deamination")
    print("detection -- it should be reframed as another instance of the same")
    print("class-weighting interaction documented for rare-CpG.")
print("=" * 100)


weighted   hotspot-CpG: mcc=0.0893  ct_recall=0.0000
unweighted hotspot-CpG: mcc=0.0610  ct_recall=0.0371

MCC collapsed (<= 0.05): False
C>T recall degenerate (>= 0.98, i.e. majority-reversion): False

DETERMINATION: (a) -- hotspot-CpG MCC survives unweighted training, undegenerate.
This is the one result in this study that survives the weighting stress-test
independent of majority-class reversion, and is the paper's most defensible
positive finding.
